In [ ]:
# create a csv that contains only animals with ephys recordings

import pandas as pd

src = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_20260120.csv"
dst = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_LFP_20260204.csv"

df = pd.read_csv(src)
df_out = df[df["id"].isin(["AM", "AN", "AO"])]
df_out.to_csv(dst, index=False)

In [ ]:
# add rows for breaks (B), add rows for ephys recordings

path_in = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_LFP_20260204.csv"
path_out = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_LFP_20260205.csv"

df = pd.read_csv(path_in)

rows = []
n = len(df)
for i, row in df.iterrows():
    rows.append(row)
    skip = (row["trial_type"] == "H") and (str(row["trial_type_day"]) in ["1", "2"])
    if not skip:
        new_row = row.copy()
        new_row["trial_type"] = "B"
        rows.append(new_row)

    if i < n - 1:
        cur_day = str(row["trial_type_day"])
        next_row = df.loc[i + 1]
        next_day = str(next_row["trial_type_day"])
        if cur_day != next_day:
            skip_extra = (next_row["trial_type"] == "H") and (str(next_row["trial_type_day"]) in ["1", "2"])
            if not skip_extra:
                extra_row = next_row.copy()
                extra_row["trial_id"] = 0
                extra_row["trial_type"] = "B"
                rows.append(extra_row)

df_out = pd.DataFrame(rows).reset_index(drop=True)

# Add ephys_trial_id per session (starts at 1 for each ses)
df_out["ephys_trial_id"] = df_out.groupby("ses").cumcount() + 1

# Insert ephys_trial_id between usable and trial_note
cols = df_out.columns.tolist()
if "ephys_trial_id" in cols:
    cols.remove("ephys_trial_id")
if "trial_note" in cols:
    insert_idx = cols.index("trial_note")
    cols.insert(insert_idx, "ephys_trial_id")
    df_out = df_out[cols]

df_out.to_csv(path_out, index=False)

In [ ]:
# move ephys recording files into the neuro blueprint format folders
from pathlib import Path
import re
import shutil

src_root = Path("/Users/stella/Desktop/PhD/paulsen_lab/Analysis/LFP_analysis/ephys")
dst_root = Path("/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/derivatives")
animal_sub = {"AM": "sub-016", "AN": "sub-017", "AO": "sub-018"}

date_re = re.compile(r"(\d{4}-\d{2}-\d{2})")
animal_re = re.compile(r"\b(AM|AN|AO)\b")
record_node_re = re.compile(r"Record Node \d+")

# NOTE: This only copies data. Source folders/files are not modified or moved.
for src_session in sorted(src_root.iterdir()):
    if not src_session.is_dir():
        continue
    if "early_test" in src_session.name:
        continue

    m_date = date_re.search(src_session.name)
    m_animal = animal_re.search(src_session.name)
    if not m_date or not m_animal:
        continue

    date_str = m_date.group(1)  # YYYY-MM-DD
    date_compact = date_str.replace("-", "")
    animal = m_animal.group(1)

    src_data = None
    for rn_dir in src_session.iterdir():
        if rn_dir.is_dir() and record_node_re.fullmatch(rn_dir.name):
            candidate = rn_dir / "experiment1"
            if candidate.is_dir():
                src_data = candidate
                break
    if src_data is None:
        continue

    sub = animal_sub.get(animal)
    if sub is None:
        continue
    animal_root = dst_root / f"{sub}_id-{animal}"
    if not animal_root.is_dir():
        continue

    target_session = None
    for ses_dir in animal_root.iterdir():
        if ses_dir.is_dir() and f"date-{date_compact}" in ses_dir.name:
            target_session = ses_dir
            break
    if target_session is None:
        continue

    ephys_dir = target_session / "ephys"
    ephys_dir.mkdir(parents=True, exist_ok=True)

    shutil.copytree(src_data, ephys_dir, dirs_exist_ok=True, copy_function=shutil.copy2)

print("Done")

Done


In [ ]:
# make sure animals have right number of recording folders, if not, go back and match them

from pathlib import Path
import re
import pandas as pd

root = Path("/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/derivatives")
animal_sub = {"AM": "sub-016", "AN": "sub-017", "AO": "sub-018"}
date_re = re.compile(r"date-(\d{8})")

records = []
warnings = []

for animal, sub in animal_sub.items():
    animal_root = root / f"{sub}_id-{animal}"
    if not animal_root.is_dir():
        continue
    for ses_dir in sorted(animal_root.iterdir()):
        if not ses_dir.is_dir():
            continue
        m = date_re.search(ses_dir.name)
        if not m:
            continue
        date_compact = m.group(1)
        date_formatted = f"{date_compact[:4]}-{date_compact[4:6]}-{date_compact[6:]}"
        ephys_dir = ses_dir / "ephys"
        if not ephys_dir.is_dir():
            count = 0
        else:
            count = sum(1 for p in ephys_dir.iterdir() if p.is_dir() and p.name != "early_test")
        records.append({
            "animal": animal,
            "date": date_formatted,
            "n_recordings": count
        })
        if count != 17:
            warnings.append((ephys_dir, count))

df_counts = pd.DataFrame(records).sort_values(["animal", "date"]).reset_index(drop=True)
display(df_counts)

for path, count in warnings:
    print(f"WARNING: {path} has {count} folders")

,animal,date,n_recordings
0,AM,2025-07-25,0
1,AM,2025-07-26,0
2,AM,2025-07-27,17
3,AM,2025-07-28,17
4,AM,2025-07-29,17
...,...,...,...
139,AO,2025-10-08,17
140,AO,2025-10-09,17
141,AO,2025-10-10,17
142,AO,2025-10-11,17
